In [1]:
from dotenv import  load_dotenv
load_dotenv()

True

In [2]:
import re
import math
import  numpy as np
import pandas as pd
from collections import Counter

DATA_URL="products.csv"
df=pd.read_csv(DATA_URL,sep="|")
print("row shape",df.shape)
df.head(3)


row shape (751, 7)


,Id,Name,Description,Price,PriceCurrency,SupplyAbility,MinimumOrder
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe,"When it comes to lightweight speed, nothing cr...",600.93,USD,396,574
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...,About this item Heavy duty construction and ru...,476.98,USD,288,993
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...,This extraordinary fusion is designed to nouri...,216.74,EUR,640,707


In [3]:
df['Description']=df["Description"].fillna("")
df['text']=df['Name']+":"+df['Description']
print("empty descriptions",(df['Description']=="").sum())
print("products:",len(df))
df[["Id","text"]].head(3)

empty descriptions 2
products: 751


,Id,text
0,d2559c95-bd28-49d8-b53a-538c34a25bcb,Saucony Men's Kinvara 13 Running Shoe:When it ...
1,c0717df8-fd27-4355-9c06-a8ad1ebb3e95,Accutire MS-4021B Digital Tire Pressure Gauge ...
2,29ea5156-8582-4345-8672-65a8f4c1e30c,SAURA LIFE SCIENCE Adivasi Ayurvedic Neelgiri ...


In [4]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())
corpus_tokens=[tokenize(t) for t in df['text']]
example=df.iloc[13]['text']
print("RAW Text")
print("",example[:110],"...")
print("TOKENS")
print("",tokenize(example)[:16],"..")
print()
print("total documents tokenized",len(corpus_tokens))

RAW Text
 Sony WH-1000XM5:Sony WH-1000XM5 headphones provide industry-leading noise cancellation, superior sound quality ...
TOKENS
 ['sony', 'wh', '1000xm5', 'sony', 'wh', '1000xm5', 'headphones', 'provide', 'industry', 'leading', 'noise', 'cancellation', 'superior', 'sound', 'quality', 'and'] ..

total documents tokenized 751


In [5]:
tokens=["hey","hey","How","are","you","doing","How","is","your","stay"]
print(Counter(tokens))

Counter({'hey': 2, 'How': 2, 'are': 1, 'you': 1, 'doing': 1, 'is': 1, 'your': 1, 'stay': 1})


In [6]:
doc_len_list=[len(tokens) for tokens in corpus_tokens]
doc_lengths=np.array(doc_len_list)
avgdl=doc_lengths.mean()
print(f"shortest document :{doc_lengths.min()} tokens")
print(f"longest document :{doc_lengths.max()} tokens")
print(f"average document length:{avgdl:.2f} tokens")

DOC_ID=13
doc_tokens=corpus_tokens[DOC_ID]
tf_counts=Counter(doc_tokens)
print(f"{DOC_ID}:{df.iloc[DOC_ID]['Name']}")
print(f"{len(doc_tokens)}tokens,{len(tf_counts)} distinict words")
pd.DataFrame(tf_counts.most_common(8),columns=["term","tf"])

shortest document :7 tokens
longest document :241 tokens
average document length:76.76 tokens
13:Sony WH-1000XM5
26tokens,22 distinict words


,term,tf
0,sony,2
1,wh,2
2,1000xm5,2
3,and,2
4,headphones,1
5,provide,1
6,industry,1
7,leading,1


In [11]:
doc_freq=Counter()
for toks in corpus_tokens:
    doc_freq.update(set(toks))
doc_freq

Counter({'and': 637,
         'the': 582,
         'for': 535,
         'a': 466,
         'with': 452,
         'to': 396,
         'of': 352,
         'is': 296,
         'in': 278,
         's': 226,
         'your': 215,
         'this': 195,
         'you': 182,
         'on': 174,
         'or': 155,
         'design': 153,
         'are': 149,
         '2': 148,
         'from': 143,
         'it': 141,
         'that': 137,
         'an': 131,
         '1': 130,
         'easy': 123,
         'high': 122,
         'can': 121,
         'quality': 111,
         '4': 107,
         '5': 107,
         'made': 107,
         '3': 104,
         'inch': 104,
         'all': 103,
         'perfect': 101,
         'more': 99,
         'our': 97,
         'use': 97,
         'women': 95,
         'offers': 93,
         'features': 92,
         'x': 87,
         'black': 86,
         'about': 84,
         'designed': 83,
         'up': 82,
         'be': 80,
         'long': 80,
         'b

In [28]:
# --- Document frequency: in how many DIFFERENT products does each word appear? ---
# This is NOT the total number of occurrences. BM25 only cares whether a
# document contains the word at least once, because a word that shows up in
# almost every product (like "the") can't help you tell products apart.
doc_freq = Counter()
for toks in corpus_tokens:
    doc_freq.update(set(toks))        # set() collapses repeats -> count each doc once

N = len(corpus_tokens)                # total number of products

for term in ["the", "wireless", "headphones"]:
    n = doc_freq[term]
    print(f'"{term:11}" appears in {n:>4} of {N} products  ({100*n/N:5.1f}% of the catalog)')

"the        " appears in  582 of 751 products  ( 77.5% of the catalog)
"wireless   " appears in   22 of 751 products  (  2.9% of the catalog)
"headphones " appears in   11 of 751 products  (  1.5% of the catalog)


In [29]:
# Attempt 1: usefulness = N / n(t)
#   N    = total products (751)
#   n(t) = how many contain the word
# A word in 1 product scores 751; a word in every product scores ~1.
demo_terms = ["and", "the", "for", "wireless", "bluetooth", "headphones", "quietcomfort", "1000xm5"]

pd.DataFrame([
    {"term": t, "in_n_docs": doc_freq[t], "N / n(t)": round(N / doc_freq[t], 1)}
    for t in demo_terms
]).sort_values("N / n(t)")

,term,in_n_docs,N / n(t)
0,and,637,1.2
1,the,582,1.3
2,for,535,1.4
3,wireless,22,34.1
4,bluetooth,14,53.6
5,headphones,11,68.3
6,quietcomfort,2,375.5
7,1000xm5,1,751.0


In [24]:
print(N)

751


In [30]:
for t in demo_terms:
    print(t, doc_freq.get(t, "Not Found"))

and 637
the 582
for 535
wireless 22
bluetooth 14
headphones 11
quietcomfort 2
1000xm5 1


In [31]:
pd.DataFrame([
    {
        "term": t,
        "in_n_docs": doc_freq[t],
        "N / n(t)": round(N / doc_freq[t], 1),
        "ln(N / n(t))": round(math.log(N / doc_freq[t]), 2),
    }
    for t in demo_terms
]).sort_values("ln(N / n(t))")

,term,in_n_docs,N / n(t),ln(N / n(t))
0,and,637,1.2,0.16
1,the,582,1.3,0.25
2,for,535,1.4,0.34
3,wireless,22,34.1,3.53
4,bluetooth,14,53.6,3.98
5,headphones,11,68.3,4.22
6,quietcomfort,2,375.5,5.93
7,1000xm5,1,751.0,6.62


In [32]:
# Attempt 2: take the logarithm of the ratio.
# A logarithm pulls large numbers down hard and small numbers down gently -
# exactly what we want: keep the ordering, kill the runaway spread.
pd.DataFrame([
    {
        "term": t,
        "in_n_docs": doc_freq[t],
        "N / n(t)": round(N / doc_freq[t], 1),
        "ln(N / n(t))": round(math.log(N / doc_freq[t]), 2),
    }
    for t in demo_terms
]).sort_values("ln(N / n(t))")

,term,in_n_docs,N / n(t),ln(N / n(t))
0,and,637,1.2,0.16
1,the,582,1.3,0.25
2,for,535,1.4,0.34
3,wireless,22,34.1,3.53
4,bluetooth,14,53.6,3.98
5,headphones,11,68.3,4.22
6,quietcomfort,2,375.5,5.93
7,1000xm5,1,751.0,6.62


In [33]:
def idf(term):
    """How much weight a word deserves, based on how rare it is.
    Higher = rarer = more useful for telling products apart.

    This is the production form. The (N - n + 0.5)/(n + 0.5) and the
    outer 1 + ... exist only to keep the number well-behaved in the two
    edge cases from the markdown above - the shape is still just "log of
    a rareness ratio", i.e. Attempt 2.
    """
    n = doc_freq.get(term, 0)
    return math.log(1 + (N - n + 0.5) / (n + 0.5))

# Give every distinct word a fixed integer id (its "slot number"). We'll need
# this in 2.5 when a document becomes a vector - each word maps to one dimension.
vocabulary = sorted(doc_freq)
term_to_id = {term: i for i, term in enumerate(vocabulary)}

print(f"corpus: {N} documents, {len(vocabulary):,} distinct terms\n")

# Compare our simple Attempt-2 formula against the production one, side by side.
# The point of this table is that they barely differ - the smoothing is cosmetic.
pd.DataFrame([
    {
        "term": t,
        "in_n_docs": doc_freq[t],
        "simple: ln(N/n)": round(math.log(N / doc_freq[t]), 2),
        "production idf": round(idf(t), 2),
    }
    for t in demo_terms
]).sort_values("production idf")

corpus: 751 documents, 8,468 distinct terms



,term,in_n_docs,simple: ln(N/n),production idf
0,and,637,0.16,0.17
1,the,582,0.25,0.26
2,for,535,0.34,0.34
3,wireless,22,3.53,3.51
4,bluetooth,14,3.98,3.95
5,headphones,11,4.22,4.18
6,quietcomfort,2,5.93,5.71
7,1000xm5,1,6.62,6.22
